-----------------------------------------------------------------------------
1. IMPORTACIONES Y RUTAS DEL PROYECTO
-----------------------------------------------------------------------------
Por cada cambio que hagan en el código o módulo que añadan, escriban un
comentario para entender qué es lo que hace su código / función, por qué
decidieron implementar esta solución, etc.

Sugerencia: definir rutas relativas a la raíz del repositorio hacia
`data/raw_data/` (lectura) y `data/procesamiento_data/` (escritura).

In [61]:
import pandas as pd
raw_data_path = '../data/raw_data'
process_data = '../data/procesamiento_data'

-----------------------------------------------------------------------------
2. CARGA DE DATOS CRUDOS (data/raw_data/)
-----------------------------------------------------------------------------
Espacio para funciones de carga de:
- transactions_data.csv (considerar lectura por bloques o columnas necesarias)
- cards_data.csv
- users_data.csv
- mcc_codes.json
- train_fraud_labels.json
Recuerda comentar qué hace cada función y por qué se implementó de esa forma.

In [62]:
user_data = pd.read_csv(raw_data_path+'/users_data.csv')
card_data = pd.read_csv(raw_data_path+'/cards_data.csv')
transcations_data = pd.read_csv(raw_data_path+'/transactions_data.csv')
mcc_codes = pd.read_json(raw_data_path+'/mcc_codes.json',typ='series').to_frame()
fraud_labes = pd.read_json(raw_data_path+'/train_fraud_labels.json')

In [63]:
display(user_data.head(1))
display(card_data.head(1))
display(transcations_data.head(1))
display(mcc_codes.head(1))
display(fraud_labes.head(1))

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5


,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN


,0
5812,Eating Places and Restaurants


,target
10649266,No


-----------------------------------------------------------------------------
3. LIMPIEZA Y TRANSFORMACION DE COLUMNAS
-----------------------------------------------------------------------------
Espacio para limpieza de datos, por ejemplo:
- Conversión de columnas monetarias con signo '$' a formato numérico.
- Conversión de fechas (`date`, `acct_open_date`) y filtrado de ventana
- temporal (5 años: 2014-2019 según lo acordado en el pizarrón).
- Tratamiento de valores nulos, duplicados y tipos de datos.
Por cada regla de limpieza que añadan, escriban un comentario explicando
qué hace y por qué tomaron esa decisión.

In [64]:
# Cambiar index de los dataframes
user_data.set_index('id',inplace=True)
card_data.set_index('id',inplace=True)
transcations_data.set_index('id',inplace=True)

# Cambiar columnas para mejor Indexacion
mcc_codes.rename(columns={0:'category'},inplace=True)
mcc_codes.rename_axis('Code',inplace=True)
fraud_labes.rename(columns={'target':'fraud'},inplace=True)
fraud_labes.rename_axis('Code',inplace=True)
user_data.rename_axis('client_id')
card_data.rename_axis('card_id')

# Cambiar transcations_data["amount"] a float
transcations_data['amount'] = pd.to_numeric(transcations_data["amount"].str.replace("$", "", regex=False), errors="coerce")

# Agregar ONLINE a 'merchant_state' y para 'zip' hacerlo 0, para cuando 'merchant_city' es ONLINE
transcations_data.loc[transcations_data['merchant_city'] == 'ONLINE', ['merchant_state']] = 'ONLINE'
transcations_data.loc[transcations_data['merchant_city'] == 'ONLINE', ['zip']] = 0

# Cambiar transcations_data["date"] a datetime
transcations_data['date'] = pd.to_datetime(transcations_data['date'],format='%Y-%m-%d %H:%M:%S')

# Display transcation_data
display(transcations_data.head(5))

,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
id,,,,,,,,,,,
7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


-----------------------------------------------------------------------------
4. INTEGRACION DE TABLAS (SEGUN docs/relaciones_tablas.md)
----------------------------------------------------------------------------
Espacio para unir tablas mediante sus llaves (`client_id`, `card_id`, `mcc`,
`transaction_id`) si el equipo decide generar tablas ya integradas en
`data/procesamiento_data/`.
Documenta con comentarios qué tipo de unión (left, inner, etc.) elegiste y por qué.

In [65]:
# Add card information to transcations_data
card_type = card_data['card_type'].to_dict()
transcations_data['card_type'] = transcations_data['card_id'].map(card_type)

card_brand = card_data['card_brand'].to_dict()
transcations_data['card_brand'] = transcations_data['card_id'].map(card_brand)

# Agregar 'purchase_category' a transcations_data
mcc_dict = mcc_codes['category'].to_dict()
transcations_data['purchase_category'] = transcations_data['mcc'].map(mcc_dict)

# Agregar 'is_fraud' a transcations_data
id_fraud = fraud_labes[fraud_labes["fraud"]=="Yes"].index
transcations_data["is_fraud"] = transcations_data.index.isin(id_fraud)

# Display transcation_data
display(transcations_data.head(5))

,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,card_type,card_brand,purchase_category,is_fraud
id,,,,,,,,,,,,,,,
7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN,Debit (Prepaid),Mastercard,Miscellaneous Food Stores,False
7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN,Credit,Mastercard,Department Stores,False
7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN,Debit,Mastercard,Money Transfer,False
7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN,Debit,Mastercard,Money Transfer,False
7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN,Debit,Visa,Drinking Places (Alcoholic Beverages),False


-----------------------------------------------------------------------------
5. EXPORTACION DE DATOS PROCESADOS (data/procesamiento_data/)
-----------------------------------------------------------------------------
Espacio para guardar los archivos limpios en `data/procesamiento_data/`
(por ejemplo en formato .csv o .parquet, ignorados automáticamente por Git).
Comenta qué archivos se generan y qué notebook de `notebooks_preguntas/` los consume.


In [67]:
transcations_data.to_csv(process_data+"/transcations_data.csv", index=True)